# 05. Impurities, LDOS and vortex-core views

Explain the supplied LDOS program, its corrected adaptation, and the saved diagrams. Original and adapted code are labelled separately.

## 1. Files, scope and how to run

The supplied [`vortex_ldos_original.m`](https://github.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/blob/main/original_code/vortex_ldos_original.m) is preserved byte for byte. The runnable adaptation is [`scripts/vortex_ldos.wl`](https://github.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/blob/main/scripts/vortex_ldos.wl). Each adapted block below comes directly from that file.

**Verified source behaviour:** the supplied program imports five matrices, adds boundary phases, diagonalizes a matrix at each momentum, computes broadened spectral sums and exports LDOS. It does not run an impurity sweep or generate the saved spatial maps.

**Adaptations:** the new file fixes the pairing-block extraction, uses a consistent hole-momentum convention, supplies explicit energy coordinates, separates sites, uses one normalization, adds impurity potentials and supplies plot functions. An optional pairing iteration is adapted from `added vortex.nb`.

**Not yet verified:** the exact historical settings and input matrices for every archived curve. Saved images remain reference results; adapted code is not described as the original generator of those images. The historical curves include conflicting unit and axis labels, so new plots use model energy units unless a conversion is independently established.

Load the `.wl` file in Mathematica using `Get["/absolute/path/to/repository/scripts/vortex_ldos.wl"]`. Loading defines functions without starting a calculation. These `.ipynb` cells are Markdown reading cells with Mathematica code, not Python execution cells. See the runnable examples in `scripts/ldos_examples.wl`.

### Setup and site numbering

`NN = Nx Ny` is the number of sites. With the saved loop order, $i=(x-1)N_y+y$. Thus site 190 is $(10,10)$ on a $20\times20$ grid, site 180 is $(9,20)$, and site 1 is $(1,1)$. Calling site 180 “nearby” is a historical label; its actual coordinate distance from the core is not one lattice spacing. Site 1 is an explicit example for the farther curve, not a verified historical choice.

`bdgRequire` stops on a failed condition and returns a labelled failure through `Catch`. `bdgNorm` is the largest absolute matrix entry, used for numerical consistency checks.

**Adapted implementation — `setup`**

```mathematica
(* SECTION: setup *)
$BdGProjectRoot = DirectoryName[DirectoryName[$InputFileName]];
ClearAll[bdgRequire, bdgNorm, SiteNumber, SiteCoordinates];
bdgRequire[test_, message_] := If[! TrueQ[test],
  Throw[Failure["BdGInput", <|"MessageTemplate" -> message|>]]];
bdgNorm[a_] := Max[Abs[Flatten[Normal[a]]]];
SiteNumber[{x_, y_}, ny_] := (x - 1) ny + y;
SiteCoordinates[i_, ny_] := {Quotient[i - 1, ny] + 1, Mod[i - 1, ny] + 1};
```

## 2. Import the normal and pairing matrices

`Huu` and `Hdd` are the two diagonal spin blocks; `Hud` and `Hdu` connect them. Each must be `NN × NN`. `SCs` is `2 NN × 2 NN`.

In the saved source, `SCs` has the form `{{0,D},{D,0}}`, where `D` is the diagonal matrix of onsite complex pairing values. The supplied LDOS file instead reads its two diagonal blocks; that would discard pairing for this layout. `LoadBdGModel` reads the upper-right block, verifies the entire layout, and sets `Delta = Diagonal[D]`.

The checks reject incompatible dimensions, non-numerical entries, non-Hermitian normal blocks and non-onsite pairing. A square matrix alone does not determine the two lattice dimensions. Supply `Nx` and `Ny` explicitly. Imported matrices do not carry a verified history of impurity settings.

**Adapted implementation — `import`**

```mathematica
(* SECTION: import *)
ClearAll[LoadBdGModel];
LoadBdGModel[folder_, nx_Integer, ny_Integer] := Module[
  {n = nx ny, names, blocks, sc, zero, d, coords},
  bdgRequire[nx >= 3 && ny >= 3, "Each lattice dimension must be at least 3."];
  names = {"Huu", "Hud", "Hdu", "Hdd", "SCs"};
  bdgRequire[And @@ (FileExistsQ[FileNameJoin[{folder, # <> ".wdx"}]] & /@ names),
    "Five WDX files are required in the input folder."];
  blocks = AssociationThread[names, Import[FileNameJoin[{folder, # <> ".wdx"}]] & /@ names];
  bdgRequire[And @@ (MatrixQ[Normal[blocks[#]], NumericQ] & /@ names), "Non-numeric matrix."];
  bdgRequire[And @@ (Dimensions[blocks[#]] === {n, n} & /@ Take[names, 4]),
    "Normal blocks do not match Nx Ny; do not guess the lattice shape."];
  sc = Normal[blocks["SCs"]];
  bdgRequire[Dimensions[sc] === {2 n, 2 n}, "SCs must have size 2 NN by 2 NN."];
  zero = ConstantArray[0., {n, n}];
  d = sc[[1 ;; n, n + 1 ;; 2 n]];
  bdgRequire[bdgNorm[sc - ArrayFlatten[{{zero, d}, {d, zero}}]] < 10^-10,
    "Expected the symmetric off-diagonal pairing layout used by the original files."];
  bdgRequire[bdgNorm[d - DiagonalMatrix[Diagonal[d]]] < 10^-10,
    "This workflow supports onsite pairing only."];
  bdgRequire[bdgNorm[blocks["Huu"] - ConjugateTranspose[blocks["Huu"]]] < 10^-10 &&
    bdgNorm[blocks["Hdd"] - ConjugateTranspose[blocks["Hdd"]]] < 10^-10 &&
    bdgNorm[blocks["Hdu"] - ConjugateTranspose[blocks["Hud"]]] < 10^-10,
    "Imported normal blocks are not Hermitian partners."];
  coords = SiteCoordinates[#, ny] & /@ Range[n];
  Join[blocks, <|"Nx" -> nx, "Ny" -> ny, "NN" -> n,
    "Coordinates" -> coords, "Delta" -> Diagonal[d], "InputFolder" -> folder|>]
];
```

## 3. Put an impurity into the electron blocks

Choose a site $i_0$ and let $P$ contain one at matrix position $(i_0,i_0)$ and zero elsewhere. Let $V$ be the impurity energy.

| Case | Change to `Huu` | Change to `Hdd` |
|---|---|---|
| No additional impurity | 0 | 0 |
| Scalar / non-magnetic | $+VP$ | $+VP$ |
| Magnetic, diagonal in the chosen spin basis | $+VP$ | $-VP$ |

The scalar case shifts both spin energies equally. The magnetic case shifts them with opposite signs. The hole block is constructed later from these changed electron blocks, so an independent hole potential is not added by hand.

All terms have energy units; $P$ is dimensionless. Setting $V=0$ recovers the input blocks exactly. A new case must start from an impurity-free baseline, not from the output of the previous impurity case. The runner requires `CleanBaselineConfirmed -> True` when an impurity is added; this is a data-provenance assertion, not a test performed by the code.

**Adapted implementation — `impurities`**

```mathematica
(* SECTION: impurities *)
ClearAll[AddImpurity];
AddImpurity[model_, kind_, strength_, site_Integer] := Module[{n = model["NN"], p},
  bdgRequire[MemberQ[{"none", "scalar", "magnetic"}, kind], "Unknown impurity kind."];
  bdgRequire[1 <= site <= n && NumericQ[strength] && TrueQ[Im[strength] == 0], "Invalid impurity."];
  p = SparseArray[{{site, site} -> N[strength]}, {n, n}];
  Join[model, <|"Huu" -> model["Huu"] + If[kind === "none", 0 p, p],
    "Hdd" -> model["Hdd"] + Switch[kind, "scalar", p, "magnetic", -p, _, 0 p],
    "ImpurityKind" -> kind, "ImpurityStrength" -> strength, "ImpuritySite" -> site|>]
];
```

## 4. Boundary phases and the full matrix

**Conventions and assumptions:** square-lattice coordinates are counted in lattice spacings ($a=1$); boundaries are periodic with Bloch twists. The imported hopping is onsite, nearest-neighbour or diagonal nearest-neighbour as in the original dictionary. Long-range hopping needs a different image-selection rule. No spacetime metric or Planck-unit convention is used.

`BoundaryPhases` stores `-1`, `0` or `+1` for an edge-crossing displacement. It follows the transposed image-lookup orientation of the supplied program. Reversing the pair of sites reverses the sign. `Exp[I(kx Nx Phasex + ky Ny Phasey)]` multiplies entries elementwise; it is not matrix multiplication. The exponent is dimensionless because each wave number has inverse-lattice-length units.

**Adapted implementation — `boundary`**

```mathematica
(* SECTION: boundary *)
ClearAll[BoundaryPhases];
BoundaryPhases[model_] := Module[{c = model["Coordinates"], n = model["NN"], lengths},
  lengths = {model["Nx"], model["Ny"]};
  Table[Table[With[{d = c[[j, axis]] - c[[i, axis]]},
    If[Abs[d] > lengths[[axis]]/2, Sign[d], 0]], {i, n}, {j, n}], {axis, 2}]
];
```

### Electron block, hole block and pairing

Let $h(k)$ be the two-spin electron matrix after the boundary phases are applied, $S=\mathrm{diag}(I,-I)$, and $B=\begin{pmatrix}0&D\\D&0\end{pmatrix}$. The adaptation uses

$$H(k)=\begin{pmatrix}h(k)&B\\B^\dagger&-S h(-k)^* S\end{pmatrix}.$$

Here $I$ is the `NN × NN` identity matrix, $*$ denotes complex conjugation, and $\dagger$ denotes conjugate transpose. The first two groups of `NN` components are electron amplitudes; the last two are hole amplitudes in the stored sign convention. Multiplying the second hole group by minus one converts this to the usual antisymmetric onsite pairing block. At $k=0$ the hole-block signs match the saved matrix layout.

The `-k` argument matters: conjugation reverses the boundary phase. Building both sectors consistently gives the hole hopping the same real-space boundary twist after conjugation. This is a correction to the supplied script's use of `Conjugate[Huuc]` at the same momentum. `ConjugateTranspose[B]` enforces the Hermitian partner for complex pairing. Onsite pairing has no edge-crossing phase.

`shift` adds a common onsite energy to the electron blocks, with the sign convention used by the original chemical-potential update. Every block of $H$ has the same energy units. At zero pairing, the electron and hole sectors decouple. At zero twist, the boundary factors become one.

**Adapted implementation — `matrix`**

```mathematica
(* SECTION: matrix *)
ClearAll[BdGMatrix];
BdGMatrix[model_, delta_, {kx_, ky_}, phases_, shift_: 0.] := Module[
  {n = model["NN"], phase, normal, minusNormal, s, zero, d, pair, h},
  phase = Exp[I (kx model["Nx"] phases[[1]] + ky model["Ny"] phases[[2]])];
  normal[p_] := ArrayFlatten[{
    {(Normal[model["Huu"]] + shift IdentityMatrix[n]) p, Normal[model["Hud"]] p},
    {Normal[model["Hdu"]] p, (Normal[model["Hdd"]] + shift IdentityMatrix[n]) p}}];
  minusNormal = normal[Conjugate[phase]];
  s = DiagonalMatrix[Join[ConstantArray[1., n], ConstantArray[-1., n]]];
  zero = ConstantArray[0., {n, n}];
  d = DiagonalMatrix[delta];
  pair = ArrayFlatten[{{zero, d}, {d, zero}}];
  h = ArrayFlatten[{{normal[phase], pair},
    {ConjugateTranspose[pair], -s . Conjugate[minusNormal] . s}}];
  bdgRequire[bdgNorm[h - ConjugateTranspose[h]] < 10^-9, "BdG matrix is not Hermitian."];
  N[h]
];
```

## 5. Construct or retain a vortex profile

With `SeedVortex -> False`, the imported pairing field is retained. With `SeedVortex -> True`, its largest magnitude supplies the amplitude scale and `VortexFactor` supplies a new, explicitly reconstructed seed.

For distance $r$ from the chosen core and positive length $\xi$, measured in lattice sites, the factor is

$$f(r)=\tanh(r/\xi)\exp(i\phi).$$

Both $r/\xi$ and $\phi$ are dimensionless. At the core, $r=0$ gives $f=0$. Far from the core, $\tanh(r/\xi)\to1$, so the magnitude approaches one. The complex exponential changes phase but not magnitude.

`Winding` uses `ArcTan[dx,dy]` to retain quadrant information. `OriginalAbsoluteAngle` retains the original use of absolute coordinate differences, which loses quadrant signs and does not describe a full $2\pi$ phase winding. These choices are labelled rather than silently substituted. The default length `.3/Pi` is inherited numerically from the saved code and is treated as a length in lattice sites; no nanometre conversion is assumed.

This is an imposed profile on a periodically repeated cell. It does not include magnetic Peierls phases, flux quantization, or a separately validated magnetic-periodic single-vortex construction.

**Adapted implementation — `vortex`**

```mathematica
(* SECTION: vortex *)
ClearAll[VortexFactor];
VortexFactor[model_, core_Integer, coherenceSites_, phaseMode_: "Winding"] := Module[
  {centre, offsets},
  bdgRequire[1 <= core <= model["NN"] && coherenceSites > 0, "Invalid vortex centre or length."];
  bdgRequire[MemberQ[{"Winding", "OriginalAbsoluteAngle"}, phaseMode], "Unknown phase mode."];
  centre = SiteCoordinates[core, model["Ny"]];
  offsets = (# - centre) & /@ model["Coordinates"];
  Map[Function[r, If[r === {0, 0}, 0.,
    Tanh[Norm[r]/coherenceSites] Exp[I If[phaseMode === "Winding",
      ArcTan[r[[1]], r[[2]]],
      If[r[[1]] == 0, Pi/2, ArcTan[Abs[r[[2]]]/Abs[r[[1]]]]]]]]], offsets]
];
```

## 6. Allow the pairing field to respond to the impurity

**Imported mode** changes the Hamiltonian while keeping the supplied pairing field fixed. Consequently its pairing maps cannot develop impurity-dependent distortions. This mode is suitable for a fixed-field spectral comparison.

**Relax mode** adapts the saved constrained-vortex iteration. At each step it:

1. Builds and diagonalizes the zero-twist matrix using the current pairing and common onsite shift.
2. Evaluates the saved clipped occupation function `f` for all eigenvalues.
3. Forms the density from the same component weights as the original solver.
4. Forms the original onsite anomalous sum, multiplies by the coupling and imposed vortex factor, and obtains `proposal`.
5. Measures the maximum difference `Abs[proposal-delta]`, mixes old and proposed fields equally, and updates the common onsite shift.
6. Stops only when both pairing and density residuals meet their tolerances, or when the iteration limit is reached.

The occupation and eigenvector weights are dimensionless. Coupling times the anomalous sum has energy units, as do `proposal`, `delta`, the pairing tolerance and onsite shift. The coefficient `.1` in the shift update carries the model energy scale. Density tolerance is dimensionless. The occupation cutoff `.02` and `kT` use the same energy unit as the matrix.

This retains the saved update convention; it is not a new derivation of a fully unconstrained vortex solution. The vortex factor is imposed at every iteration. The momentum average is applied afterward to LDOS, as in the supplied workflow. Reaching 100 iterations is not evidence of convergence. `RelaxPairing` returns the iteration history even if unconverged; the high-level runner refuses to label an unconverged result as a completed case. The final spectrum and LDOS are rebuilt from the same final field and onsite shift.

**Adapted implementation — `relaxation`**

```mathematica
(* SECTION: relaxation *)
ClearAll[RelaxPairing];
RelaxPairing[model_, initial_, factor_, phases_, settings_] := Module[
  {delta = N[initial], shift = 0., h, energies, vectors, f, proposal,
   next, nextShift, density, residual, history = {}, converged = False,
   n = model["NN"], temperature = settings["kT"], j},
  bdgRequire[temperature > 0 && settings["MaxIterations"] > 0, "Invalid iteration settings."];
  Do[
    h = BdGMatrix[model, delta, {0., 0.}, phases, shift];
    {energies, vectors} = Eigensystem[h];
    (* Same clipped Fermi function, component pairing sum and mixing as the saved solver. *)
    f = (If[# > .02, 0., If[# < -.02, 1., 1./(Exp[#/temperature] + 1.)]] &) /@ energies;
    density = Mean[Table[Abs[vectors[[All, j]]]^2 . f +
      Abs[vectors[[All, j + 3 n]]]^2 . (1. - f), {j, n}]];
    proposal = settings["Coupling"] Table[
      (vectors[[All, j]] Conjugate[vectors[[All, j + 3 n]]] +
       vectors[[All, j + n]] Conjugate[vectors[[All, j + 2 n]]]) . f, {j, n}] factor;
    residual = Max[Abs[proposal - delta]];
    next = (proposal + delta)/2.;
    nextShift = shift - .1 (settings["TargetDensity"] - density);
    AppendTo[history, {iteration, residual, density, shift}];
    converged = residual < settings["Tolerance"] &&
      Abs[density - settings["TargetDensity"]] < settings["DensityTolerance"];
    delta = next; shift = nextShift;
    If[converged, Break[]], {iteration, settings["MaxIterations"]}];
  <|"Delta" -> delta, "Shift" -> shift, "Converged" -> converged, "History" -> history|>
];
```

## 7. Calculate normalized LDOS

`Eigensystem` returns an energy list and a list of eigenvectors. The code uses **all eigenvalues**, and the electron weights of both spin groups. This avoids adding a second hole contribution to an already complete electron spectral sum.

For eigenstate $n$ and site $i$, define its spin-averaged electron weight

$$w_{ni}=\frac{|u_{n i \uparrow}|^2+|u_{n i \downarrow}|^2}{2}.$$

To see the broadening used in the supplied code, set $x=E-E_n$ and multiply numerator and denominator by the complex conjugate:

$$\frac{1}{x+i\eta}=\frac{x-i\eta}{(x+i\eta)(x-i\eta)}=\frac{x-i\eta}{x^2+\eta^2},$$
$$-\frac1\pi\operatorname{Im}\frac{1}{x+i\eta}=\frac{\eta}{\pi(x^2+\eta^2)}.$$

The adaptation therefore evaluates

$$\rho_i(E)=\frac{1}{N_s^2}\sum_{k}\sum_n w_{ni}(k)\frac{\eta}{\pi[(E-E_n(k))^2+\eta^2]}.$$

`Ns` is the number of samples along each momentum direction, so there are `Ns^2` matrices. `eta` is positive and has energy units. The weights and momentum average are dimensionless; LDOS has inverse-energy units. The output array has one row per energy and one column per selected site.

**Normalization check:** integrating one broadened peak gives

$$\int_{-\infty}^{\infty}\frac{\eta\,dE}{\pi[(E-E_n)^2+\eta^2]}=\frac1\pi\left[\arctan\frac{E-E_n}{\eta}\right]_{-\infty}^{\infty}=1.$$

For orthonormal complete eigenvectors, each electron component has total weight one across all states; the two-spin average thus gives $\sum_n w_{ni}=1$. Averaging `Ns^2` such spectra leaves $\int\rho_i(E)dE=1$. A finite plotted energy window need not contain that entire weight. The spin-summed LDOS would be twice this result and integrate to two.

As $\eta\to0^+$, peaks narrow toward delta functions; on a finite grid they require finer sampling. Far from all eigenvalues, the tails decay as $1/E^2$. `Ns=1` gives the zero-twist result. Increasing `Ns` is a sampling choice, not a normalization change.

The original script instead sums its first electron and first hole component over the full eigensystem with no factor of one half; its full-energy integral is two for a complete orthonormal basis. Its curves must not be assumed numerically identical to this explicitly defined spin average. The new implementation is serial for transparent execution; it does not close or launch the user's parallel kernels.

**Adapted implementation — `ldos`**

```mathematica
(* SECTION: ldos *)
ClearAll[CalculateLDOS];
CalculateLDOS[model_, delta_, phases_, sites_, energies_, eta_, ns_Integer, shift_: 0.] := Module[
  {n = model["NN"], rho, h, ev, vec, weights, spectral, kx, ky},
  bdgRequire[eta > 0 && ns >= 1 && Length[sites] > 0 &&
    And @@ (IntegerQ[#] && 1 <= # <= n & /@ sites), "Invalid LDOS grid or sites."];
  rho = ConstantArray[0., {Length[energies], Length[sites]}];
  Do[
    kx = 2 Pi a/(ns model["Nx"]); ky = 2 Pi b/(ns model["Ny"]);
    h = BdGMatrix[model, delta, {kx, ky}, phases, shift];
    {ev, vec} = Eigensystem[h];
    (* Full spectrum + electron components only: one state per site per spin on average. *)
    weights = (Abs[vec[[All, sites]]]^2 + Abs[vec[[All, sites + n]]]^2)/2.;
    spectral = Table[eta/(Pi ((energy - ev)^2 + eta^2)), {energy, energies}];
    rho += spectral . weights,
    {a, 0, ns - 1}, {b, 0, ns - 1}];
  rho/ns^2
];
```

## 8. Plot energy curves, spectra and vortex-core views

`LDOSCurves` supplies explicit `{energy, value}` pairs for every site. It draws green, blue and orange curves in the order supplied in `Sites`. Reference energies are optional inputs; red lines are not automatically measured gaps.

`PairingViews` uses the **same** `Abs[delta]` values for all three views:

- **Line:** a cut along x at the core's y coordinate.
- **Map:** the magnitude at every `(x,y)` coordinate, with a colour legend.
- **Surface:** the same magnitude as height, with a conventional increasing vertical axis.

These are pairing-amplitude views, not LDOS, phase, or magnetic-field maps. Their plotted heights have energy units. Their appearance need not match an archived surface with a reversed vertical axis or a different sign convention. No curve is fitted to an image.

`SpectrumView` plots sorted eigenvalues against sorted-state index. `LDOSMap` instead maps the LDOS at one selected energy across all sites; it requires `Sites -> All`. A large LDOS value at the core is a spectral-weight observation, not proof of a particular bound-state identity.

**Adapted implementation — `plots`**

```mathematica
(* SECTION: plots *)
ClearAll[LDOSCurves, PairingViews, SpectrumView, LDOSMap];
LDOSCurves[energies_, rho_, sites_, references_: {}, unit_: "model energy"] :=
 ListLinePlot[Table[Transpose[{energies, rho[[All, j]]}], {j, Length[sites]}],
  Frame -> True, FrameLabel -> {"Energy (" <> unit <> ")", "Spin-averaged LDOS (1/" <> unit <> ")"},
  PlotLegends -> ("site " <> ToString[#] & /@ sites), PlotStyle -> {Green, Blue, Orange},
  GridLines -> {references, None}, GridLinesStyle -> Directive[Red, Dotted], PlotRange -> All];
PairingViews[model_, delta_, core_Integer] := Module[{xyz, y0, line},
  xyz = MapThread[Append, {model["Coordinates"], Abs[delta]}];
  y0 = SiteCoordinates[core, model["Ny"]][[2]];
  line = Table[{x, Abs[delta[[SiteNumber[{x, y0}, model["Ny"]]]]]}, {x, model["Nx"]}];
  <|"Line" -> ListLinePlot[line, Frame -> True, FrameLabel -> {"x (lattice sites)", "|Delta|"}, PlotRange -> All],
    "Map" -> ListDensityPlot[xyz, InterpolationOrder -> 0, PlotLegends -> Automatic,
      FrameLabel -> {"x (lattice sites)", "y (lattice sites)"}, PlotRange -> All],
    "Surface" -> ListPlot3D[xyz, InterpolationOrder -> 1, Mesh -> All,
      AxesLabel -> {"x (lattice sites)", "y (lattice sites)", "|Delta|"}, PlotRange -> All]|>
];
SpectrumView[h_, references_: {}] := ListPlot[Sort[Eigenvalues[h]], Frame -> True,
  FrameLabel -> {"Sorted-state index", "Energy (model units)"}, PlotRange -> All,
  GridLines -> {None, references}, GridLinesStyle -> Directive[Green, Dashed]];
LDOSMap[model_, sites_, rho_, energyIndex_Integer] := Module[{xyz},
  bdgRequire[Sort[sites] === Range[model["NN"]], "A full spatial LDOS map needs all lattice sites."];
  xyz = MapThread[Append, {model["Coordinates"][[sites]], rho[[energyIndex]]}];
  ListDensityPlot[xyz, InterpolationOrder -> 0, PlotLegends -> Automatic,
    FrameLabel -> {"x (lattice sites)", "y (lattice sites)"}, PlotRange -> All]
];
```

## 9. Export and run an explicitly labelled case

The CSV has three columns: energy, site and spin-averaged LDOS. Keeping the site column prevents interleaved curves from being treated as one line. A WXF file retains the matrices, resulting pairing, grid, settings and metadata. A new output directory is required so an earlier result is not overwritten.

**Adapted implementation — `export`**

```mathematica
(* SECTION: export *)
ClearAll[ExportLDOS];
ExportLDOS[folder_, result_] := Module[{rows, history},
  bdgRequire[! DirectoryQ[folder], "Choose a new output folder to preserve existing results."];
  CreateDirectory[folder, CreateIntermediateDirectories -> True];
  rows = Flatten[Table[{result["Energy"][[e]], result["Sites"][[j]], result["LDOS"][[e, j]]},
    {e, Length[result["Energy"]]}, {j, Length[result["Sites"]]}], 1];
  Export[FileNameJoin[{folder, "ldos.csv"}], Prepend[rows, {"energy", "site", "spin_averaged_ldos"}], "CSV"];
  Export[FileNameJoin[{folder, "case.wxf"}], result, "WXF"];
  If[KeyExistsQ[result, "IterationHistory"],
    history = Prepend[result["IterationHistory"], {"iteration", "gap_residual", "density", "onsite_shift"}];
    Export[FileNameJoin[{folder, "iterations.csv"}], history, "CSV"]];
  folder
];
```

### Main options

| Option | Default | Meaning |
|---|---|---|
| `Nx`, `Ny` | 20, 20 | Dimensions expected in the imported data |
| `Ns` | 10 | Samples per momentum direction |
| `Wmax`, `Intervals` | 0.10, 400 | 401 energies from -0.10 to +0.10 |
| `Eta` | 0.005 | Positive broadening in model energy units |
| `Sites` | `{190,180,1}` | Explicit example observation sites |
| `ImpurityKind`, `Strength` | `"none"`, 0 | Additional impurity, if requested |
| `CoreSite` | 190 | Core/impurity site on the chosen lattice |
| `PairingMode` | `"Imported"` | Retain the field or use `"Relax"` |
| `SeedVortex` | `False` | Retain imported field or reconstruct a seed |
| `PhaseMode` | `"Winding"` | Quadrant-aware phase; original formula remains selectable |

The default does not assume the bundled WDX data contain a specific historical vortex or impurity. The matrix checks must pass before calculation. Full $20\times20$ cases involve dense $1600\times1600$ eigensystems; `Ns=20` means 400 such decompositions per LDOS case. The example file never starts a sweep just by loading it.

**Adapted implementation — `run`**

```mathematica
(* SECTION: run *)
ClearAll[RunLDOSCase];
Options[RunLDOSCase] = {"Nx" -> 20, "Ny" -> 20, "Ns" -> 10,
  "Wmax" -> .10, "Intervals" -> 400, "Eta" -> .005,
  "Sites" -> {190, 180, 1}, "ImpurityKind" -> "none", "Strength" -> 0., "CoreSite" -> 190,
  "CleanBaselineConfirmed" -> False, "PairingMode" -> "Imported",
  "SeedVortex" -> False, "CoherenceSites" -> .3/Pi, "PhaseMode" -> "Winding",
  "Coupling" -> -1.25, "kT" -> .001, "TargetDensity" -> 1., "MaxIterations" -> 100,
  "Tolerance" -> 10^-7, "DensityTolerance" -> 10^-6};
RunLDOSCase[folder_, OptionsPattern[]] := Catch[Module[
  {model, phases, delta, factor, mode, settings, relaxed, shift = 0., energy, sites,
   rho, result, kind = OptionValue["ImpurityKind"], core = OptionValue["CoreSite"]},
  bdgRequire[OptionValue["Wmax"] > 0 && IntegerQ[OptionValue["Intervals"]] &&
    OptionValue["Intervals"] > 0 && IntegerQ[OptionValue["Ns"]], "Invalid sampling parameters."];
  bdgRequire[kind === "none" || TrueQ[OptionValue["CleanBaselineConfirmed"]],
    "Confirm that input blocks are an impurity-free baseline before adding a potential."];
  model = LoadBdGModel[folder, OptionValue["Nx"], OptionValue["Ny"]];
  bdgRequire[1 <= core <= model["NN"], "Core site is outside the lattice."];
  model = AddImpurity[model, kind, OptionValue["Strength"], core];
  phases = BoundaryPhases[model]; delta = model["Delta"];
  factor = ConstantArray[1., model["NN"]];
  If[TrueQ[OptionValue["SeedVortex"]],
    factor = VortexFactor[model, core, OptionValue["CoherenceSites"], OptionValue["PhaseMode"]];
    bdgRequire[Max[Abs[delta]] > 0, "A nonzero input pairing amplitude is required to seed a vortex."];
    delta = Max[Abs[delta]] factor];
  mode = OptionValue["PairingMode"];
  bdgRequire[MemberQ[{"Imported", "Relax"}, mode], "PairingMode must be Imported or Relax."];
  bdgRequire[mode =!= "Relax" || TrueQ[OptionValue["SeedVortex"]],
    "Relax mode requires an explicit vortex seed; it applies the saved constrained-vortex update."];
  settings = <|"Coupling" -> OptionValue["Coupling"], "kT" -> OptionValue["kT"],
    "TargetDensity" -> OptionValue["TargetDensity"], "MaxIterations" -> OptionValue["MaxIterations"],
    "Tolerance" -> OptionValue["Tolerance"], "DensityTolerance" -> OptionValue["DensityTolerance"]|>;
  If[mode === "Relax",
    relaxed = RelaxPairing[model, delta, factor, phases, settings];
    bdgRequire[TrueQ[relaxed["Converged"]],
      "Pairing did not converge. Inspect RelaxPairing history; no converged LDOS result was produced."];
    delta = relaxed["Delta"]; shift = relaxed["Shift"]];
  sites = Replace[OptionValue["Sites"], All -> Range[model["NN"]]];
  energy = N[Subdivide[-OptionValue["Wmax"], OptionValue["Wmax"], OptionValue["Intervals"]]];
  rho = CalculateLDOS[model, delta, phases, sites, energy, OptionValue["Eta"], OptionValue["Ns"], shift];
  result = <|"Model" -> model, "Delta" -> delta, "Shift" -> shift, "Sites" -> sites,
    "Energy" -> energy, "LDOS" -> rho, "Ns" -> OptionValue["Ns"], "Eta" -> OptionValue["Eta"],
    "PairingMode" -> mode, "SeedVortex" -> OptionValue["SeedVortex"],
    "PhaseMode" -> OptionValue["PhaseMode"], "CoherenceSites" -> OptionValue["CoherenceSites"],
    "IterationSettings" -> settings, "CoreSite" -> core,
    "EnergyUnit" -> "model energy", "Normalization" -> "electron spin average; all eigenvalues"|>;
  If[mode === "Relax", result = Join[result, <|"IterationHistory" -> relaxed["History"]|>]];
  result
]];
```

## 10. Match the types of saved figures without inventing their inputs

| Saved figure family | Relevant code | Available setting / limitation |
|---|---|---|
| Energy-axis LDOS, including the two inner peaks | `CalculateLDOS`, `LDOSCurves` | The vortex curve is labelled with a 20 × 20 lattice and 20 supercells; other historical curves use 28 × 28 and 40. Each needs compatible input matrices. |
| Green/blue/orange impurity LDOS | `AddImpurity`, `CalculateLDOS`, `LDOSCurves` | Scalar and magnetic strengths 0.5, 5 and 50; sites 190 and 180 are identified; exact farther site is unverified. |
| Three pairing line profiles | `RelaxPairing`, `PairingViews[...]["Line"]` | Displayed strengths are 0.5, 1 and 5. Newly relaxed fields must converge separately. |
| Impurity energy spectra | `BdGMatrix`, `SpectrumView` | Diagonalize each case's own final matrix; choose reference lines explicitly. |
| Two-dimensional vortex-core views | `PairingViews[...]["Map"]` | Use the final onsite pairing magnitude and explicit coordinates. |
| Three-dimensional vortex-core views | `PairingViews[...]["Surface"]` | Same pairing data as the map; new plots label increasing height normally. |
| Optional spatial LDOS at a chosen energy | `LDOSMap` | This is a distinct observable, not a relabelling of a pairing map. |

**Axis ambiguity in the archived impurity curves:** the images say “Number of Sites”, but the supplied `ListPlot[Table[ldos[[n,184]],{n,1,nw}]]` varies the energy-row index at a fixed site. Its horizontal index therefore represents energy samples. This makes an energy-sample interpretation plausible for the old curves, but does not establish that this exact command produced every image. Preserve the archived labels and explain the ambiguity; use explicit energy coordinates for new curves.

The code below is linked to the diagram families by what it computes. Exact historical reproduction remains unverified until the corresponding converged matrices, observation sites, energy units and run settings are available. No new numerical curves are embedded as if they were old results.

## Saved energy-axis LDOS diagrams

The following are archived outputs. Their original energy labels and scales are retained; the adapted workflow has not been run to reproduce these numerical curves.

### LDOS with a narrow central peak

![LDOS with a narrow central peak](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_central_peak.png)

The blue curve has a large peak close to zero energy, with many smaller oscillations on either side. The curve remains appreciable through the central region. The small oscillations are features of the saved curve; the image alone does not establish their numerical origin.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_central_peak.png)

### LDOS suppressed between two sharp peaks

![LDOS suppressed between two sharp peaks](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_gap.png)

Red dotted reference lines lie at approximately -0.06 and +0.06 meV. Between them, the LDOS is close to zero; near either line it rises sharply. The two sides are not identical in height and background. These plotted values should be read directly rather than assumed symmetric.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_gap.png)

### Two peaks inside the marked energy interval

![Two peaks inside the marked energy interval](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_two_inner_peaks.png)

The red dotted lines mark approximately ±0.05 meV. Two broad maxima appear inside that interval, with a smaller but nonzero LDOS near zero energy. A central dip and two finite-energy peaks are different from a zero-energy peak.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_two_inner_peaks.png)

## Saved pairing line profiles

The three panels below plot pairing amplitude against position. Their horizontal coordinate is spatial, unlike the energy axes immediately above.

### Three spatial profiles with different local distortion

![Three spatial profiles with different local distortion](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_profiles_impurity.png)

All three orange curves approach zero near the central coordinate. From left to right, the neighbouring region becomes more uneven, while values far from the centre stay around the same general level. The panel scales are slightly different: the rightmost vertical scale reaches about 0.08, whereas the first two reach about 0.07. The colours identify the plotted line, not three distinct observables.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_profiles_impurity.png)

## Saved impurity LDOS curves

The next six saved images label the horizontal axis **Number of Sites**. That is the archived label. The supplied LDOS program shows how an energy-sample index can appear on this axis; the exact generating command for each saved image remains unverified. The curves use green for the impurity-site result, blue for a nearby-site result and orange for a farther-site result. The adapted code uses explicit energy values and site identifiers for new curves, resolving the ambiguity going forward.

The panels are grouped by their stored scalar/nonmagnetic or magnetic case. The descriptions below state what is visible; they do not assume that changes in peak height prove protection or identify a particular bound state.

### Scalar case: green curve above the other two

![Scalar case: green curve above the other two](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_scalar_weak.png)

The green trace lies above the blue and orange traces through much of the index range. All three curves show a deep minimum around index 100, with larger values on both sides. The green minimum remains slightly higher than the others.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_scalar_weak.png)

### Scalar case: a reduced green trace

![Scalar case: a reduced green trace](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_scalar_middle.png)

The green curve is much smaller than the other two over most indices. The blue and orange curves retain a central depression and prominent shoulders. The orange trace is higher around several shoulder peaks, so colour order is not fixed across all cases.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_scalar_middle.png)

### Scalar case: green trace close to the baseline

![Scalar case: green trace close to the baseline](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_scalar_strong.png)

The green trace is nearly flat at zero. Blue and orange retain a substantial signal away from the central dip, with the blue trace generally larger. The result is strongly position dependent; it is not a uniform rescaling of every curve.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_scalar_strong.png)

### Magnetic case: three resolved LDOS traces

![Magnetic case: three resolved LDOS traces](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_magnetic_weak.png)

All three traces have a central trough. The blue curve has the largest shoulder peaks, while green and orange cross in parts of the outer region. Compare the traces at the same horizontal index to assess their relative values.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_magnetic_weak.png)

### Magnetic case: suppression of the green trace

![Magnetic case: suppression of the green trace](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_magnetic_middle.png)

The green curve is low across the plotted range. Blue and orange show pronounced shoulders surrounding the central trough. The blue shoulders are taller, but the two outer traces remain similar in some regions.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_magnetic_middle.png)

### Magnetic case: green trace near zero

![Magnetic case: green trace near zero](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_magnetic_strong.png)

The green result stays close to the baseline. The remaining two traces still have a broad central minimum, with blue generally above orange. The vertical scale differs from other panels, so identical apparent heights on screen need not represent identical values.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_magnetic_strong.png)

## Saved impurity spectra

In each spectrum below, horizontal n is the index in a sorted energy list. Blue dots are eigenvalues. Green dashed lines are selected energy references; the saved plotting code sets their positions explicitly.

### Scalar spectrum: levels inside the reference interval

![Scalar spectrum: levels inside the reference interval](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_scalar_a.png)

Several blue dots lie between the green lines around zero. More distant levels occur above and below that interval. This plot shows energies, not the locations of their wavefunctions.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_scalar_a.png)

### Magnetic spectrum: small positive and negative levels

![Magnetic spectrum: small positive and negative levels](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_magnetic_a.png)

Blue levels occur on both sides of zero, including inside the marked interval. Closely spaced points can overlap at this image resolution, so an exact degeneracy count should not be inferred from visible dots alone.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_magnetic_a.png)

### Scalar spectrum with a near-zero level

![Scalar spectrum with a near-zero level](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_scalar_b.png)

A dot lies close to zero and other levels lie near the reference boundaries. The title records the parameters attached to this saved plot. A near-zero energy alone does not establish the state’s spatial distribution or physical identity.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_scalar_b.png)

### Magnetic spectrum with a near-zero level

![Magnetic spectrum with a near-zero level](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_magnetic_b.png)

The same types of features are visible: a central near-zero level, nearby levels and an outer sequence of states. The broad resemblance to another spectrum does not establish equality of the underlying matrices or wavefunctions.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_magnetic_b.png)

## Saved spatial maps and surfaces

Colour maps place one cell at each lattice coordinate. The surface diagrams also show a vertical numerical axis. Several of these surfaces have **zero at the top of the vertical axis and larger positive values lower down**. An apparent upward spike can therefore correspond to a smaller value; read the ticks before describing it as an enhancement.

### Scalar case: a distinct central cell

![Scalar case: a distinct central cell](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_map_scalar_weak.png)

A pale cell near (10,10) stands out from the surrounding orange and yellow cells. The coordinates span a 20 × 20 grid, with y increasing downward. There is no colour bar, so exact amplitudes cannot be recovered from the colours.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_map_scalar_weak.png)

### Scalar surface with an inverted vertical scale

![Scalar surface with an inverted vertical scale](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_surface_scalar_a.png)

The surface is uneven and a sharp central feature rises toward the tick marked zero. Larger positive tick values are lower in the drawing. Thus its visual upward direction must not be interpreted as an increase of the plotted quantity.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_surface_scalar_a.png)

### Scalar case: irregular surrounding cells

![Scalar case: irregular surrounding cells](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_map_scalar_strong.png)

The central region is pale, while surrounding cells fluctuate more strongly between yellow and orange. The map displays spatial structure but supplies no numerical colour scale. It does not show the phase of a complex field.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_map_scalar_strong.png)

### Scalar surface with a central feature and rough surroundings

![Scalar surface with a central feature and rough surroundings](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_surface_scalar_b.png)

The vertical ticks run from about 0 near the top toward 0.08 below. The central feature points toward smaller values, while the surrounding surface is uneven. Its vertical range differs from the preceding scalar surface.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_surface_scalar_b.png)

### Magnetic surface with a pronounced central feature

![Magnetic surface with a pronounced central feature](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_surface_magnetic.png)

The vertical scale again increases downward, from 0 toward about 0.06. The central feature reaches toward the smaller-value end of that scale. Away from the centre, the mesh has substantial site-to-site variation.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_surface_magnetic.png)

### Magnetic case: patchy spatial pattern

![Magnetic case: patchy spatial pattern](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_map_magnetic.png)

Orange and yellow cells alternate across much of the grid, and a pale central cell remains visible. The plot gives spatial coordinates but no numerical colour key. It supports a description of the pattern, not a precise amplitude comparison with another colour map.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_map_magnetic.png)